In [1]:
%env CUBLAS_WORKSPACE_CONFIG=:4096:8


env: CUBLAS_WORKSPACE_CONFIG=:4096:8


In [2]:
import pandas as pd
from rdkit import Chem
import torch

from sklearn.metrics import roc_auc_score
from torch_geometric.loader import DataLoader
import numpy as np

from sklearn.model_selection import train_test_split
from itertools import product
import random


import sys
import os
sys.path.append(os.path.abspath('../'))
from reduceGraph import mol_to_graph, graph_to_pyg_oh, reduce_graph_from_mol_oh, mol_to_pool_idx
from networks import GAT, PPGAT

In [3]:
#set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(mode=True)

In [5]:
#DATASET 

dataset = pd.read_csv('BBBP.csv')
dataset


,num,name,p_np,smiles
0,1,Propanolol,1,[Cl].CC(C)NCC(O)COc1cccc2ccccc12
1,2,Terbutylchlorambucil,1,C(=O)(OC(C)(C)C)CCCc1ccc(cc1)N(CCCl)CCCl
2,3,40730,1,c12c3c(N4CCN(C)CC4)c(F)cc1c(c(C(O)=O)cn2C(C)CO...
3,4,24,1,C1CCN(CC1)Cc1cccc(c1)OCCCNC(=O)C
4,5,cloxacillin,1,Cc1onc(c2ccccc2Cl)c1C(=O)N[C@H]3[C@H]4SC(C)(C)...
...,...,...,...,...
2045,2049,licostinel,1,C1=C(Cl)C(=C(C2=C1NC(=O)C(N2)=O)[N+](=O)[O-])Cl
2046,2050,ademetionine(adenosyl-methionine),1,[C@H]3([N]2C1=C(C(=NC=N1)N)N=C2)[C@@H]([C@@H](...
2047,2051,mesocarb,1,[O+]1=N[N](C=C1[N-]C(NC2=CC=CC=C2)=O)C(CC3=CC=...
2048,2052,tofisoline,1,C1=C(OC)C(=CC2=C1C(=[N+](C(=C2CC)C)[NH-])C3=CC...


In [6]:
dataset.p_np.value_counts()

p_np
1    1567
0     483
Name: count, dtype: int64

In [7]:
def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  
    graph = mol_to_graph(mol)   # mol to networkx 
    data = graph_to_pyg_oh(graph)  # networx graph to pytorch geometric graph
    data.y = torch.tensor([label], dtype=torch.float) #add label 
    return data

def dataframe_to_pyg_dataset(df, smiles_col, label_col):
    data_list = []
    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        label = row[label_col]
        data = smiles_to_data(smiles, label)
        if data is not None:
            data.smiles = smiles
            data_list.append(data)

    return data_list

In [8]:
bbbp_dataset = dataframe_to_pyg_dataset(dataset, 'smiles', 'p_np')


[10:26:18] Explicit valence for atom # 1 N, 4, is greater than permitted
[10:26:18] WARNING: not removing hydrogen atom without neighbors
[10:26:18] Explicit valence for atom # 6 N, 4, is greater than permitted
[10:26:18] WARNING: not removing hydrogen atom without neighbors
[10:26:18] WARNING: not removing hydrogen atom without neighbors
[10:26:18] WARNING: not removing hydrogen atom without neighbors
[10:26:19] WARNING: not removing hydrogen atom without neighbors
[10:26:19] WARNING: not removing hydrogen atom without neighbors
[10:26:19] WARNING: not removing hydrogen atom without neighbors
[10:26:19] Explicit valence for atom # 6 N, 4, is greater than permitted
[10:26:19] WARNING: not removing hydrogen atom without neighbors
[10:26:19] WARNING: not removing hydrogen atom without neighbors
[10:26:19] WARNING: not removing hydrogen atom without neighbors
[10:26:19] WARNING: not removing hydrogen atom without neighbors
[10:26:19] Explicit valence for atom # 11 N, 4, is greater than pe

In [9]:
#grid search
learning_rates = [1e-4, 5e-4, 1e-3]
batch_sizes = [16, 32, 64]
grid = list(product(learning_rates, batch_sizes))

In [ ]:


def train_eval_model(mod, dataset, in_channels, edge_attr_dim, out_channels, grid, epochs=30):
    labels = [int(data.y.item()) for data in dataset]

    train_val_indices, test_indices = train_test_split(
        list(range(len(dataset))),
        test_size=0.2,
        stratify=labels,
        random_state=42
    )

    train_val_labels = [labels[i] for i in train_val_indices]

    # 10% of full = 12.5% of 80% → 0.125
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.125,
        stratify=train_val_labels,
        random_state=42
    )

    # Create subsets 
    train = [dataset[i] for i in train_indices]
    val = [dataset[i] for i in val_indices]
    test  = [dataset[i] for i in test_indices]


    #  Hyperparameter tuning 
    best_val_loss = float('inf')
    best_config = None
    best_model_state = None

    for lr, batch_size in grid:
        model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to('cuda')
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.BCEWithLogitsLoss()

        train_loader = DataLoader(train, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val, batch_size=batch_size)

        for epoch in range(epochs):
            model.train()
            for batch in train_loader:
                batch = batch.to('cuda')
                optimizer.zero_grad()
                out = model(batch)
                loss = criterion(out, batch.y.float().unsqueeze(1))
                loss.backward()
                optimizer.step()

        # Evaluate on val
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to('cuda')
                out = model(batch)
                val_loss += criterion(out, batch.y.float().unsqueeze(1)).item() * batch.num_graphs


        val_loss /= len(val)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_config = (lr, batch_size)
            best_model_state = model.state_dict()

    #  Retrain final model on train + val 
    final_model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to('cuda')
    final_model.load_state_dict(best_model_state)

    full_train = train + val
    train_loader = DataLoader(full_train, batch_size=best_config[1], shuffle=True)
    optimizer = torch.optim.Adam(final_model.parameters(), lr=best_config[0])
    criterion = torch.nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        final_model.train()
        for batch in train_loader:
            batch = batch.to('cuda')
            optimizer.zero_grad()
            out = final_model(batch)
            loss = criterion(out, batch.y.float().unsqueeze(1))
            loss.backward()
            optimizer.step()

    return final_model, best_config, best_val_loss

In [11]:
in_channels = bbbp_dataset[0].x.size(-1)
edge_attr_dim = bbbp_dataset[0].edge_attr.size(-1)
out_channels = 1

model, config, loss = train_eval_model(GAT, bbbp_dataset, in_channels, edge_attr_dim, out_channels, grid)

print(config )


(0.0005, 32)


In [14]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
import torch

from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score,
    recall_score,
    average_precision_score,
    matthews_corrcoef
)


In [15]:
def cross_validate_stratified(
    mod,
    dataset, in_channels, edge_attr_dim, out_channels,
    best_lr, best_batch_size, epochs=30, k=5
):


    # Extract single target per sample
    y_vector = torch.stack([data.y.view(-1)[0] for data in dataset]).numpy()

    splitter = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    all_aurocs = []
    all_bal_accs = []
    all_f1s = []
    all_sensitivities = []
    all_specificities = []
    all_pr_aucs = []
    all_mccs = []

    for fold, (train_val_idx, test_idx) in enumerate(splitter.split(np.zeros(len(y_vector)), y_vector)):
        print(f"\n Fold {fold + 1}")

        # Split data
        y_train_val = y_vector[train_val_idx]
        train_val_data = [dataset[i] for i in train_val_idx]
        test_data = [dataset[i] for i in test_idx]

        # Inner val split
        inner_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=fold)
        train_idx, val_idx = next(inner_split.split(np.zeros(len(y_train_val)), y_train_val))
        train = [train_val_data[i] for i in train_idx]
        val = [train_val_data[i] for i in val_idx]

        # Initialize model
        model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to('cuda')
        optimizer = torch.optim.Adam(model.parameters(), lr=best_lr)
        criterion = torch.nn.BCEWithLogitsLoss()

        train_loader = DataLoader(train, batch_size=best_batch_size, shuffle=True)
        val_loader = DataLoader(val, batch_size=best_batch_size)
        test_loader = DataLoader(test_data, batch_size=best_batch_size)

        # Training loop
        for epoch in range(epochs):
            model.train()
            for batch in train_loader:
                batch = batch.to('cuda')
                optimizer.zero_grad()
                out = model(batch)

                y_true = batch.y.float()
                if y_true.dim() == 1:
                    y_true = y_true.unsqueeze(1)

                loss = criterion(out, y_true)
                loss.backward()
                optimizer.step()

        # Evaluate on test set
        model.eval()
        y_true, y_scores = [], []

        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to('cuda')
                out = model(batch)

                y_scores.append(out.cpu())
                y_true.append(batch.y.cpu())

        y_scores = torch.cat(y_scores).numpy()
        y_true = torch.cat(y_true).numpy()

        # AUROC
        auroc = roc_auc_score(y_true, y_scores)
        all_aurocs.append(auroc)

        # Convert logits → probabilities → predictions
        y_probs = 1 / (1 + np.exp(-y_scores))   # sigmoid
        y_pred = (y_probs >= 0.5).astype(int)

        # Balanced accuracy
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        all_bal_accs.append(bal_acc)

        # F1 score
        f1 = f1_score(y_true, y_pred)
        all_f1s.append(f1)

        # Sensitivity (Recall / True Positive Rate)
        sensitivity = recall_score(y_true, y_pred)
        all_sensitivities.append(sensitivity)

        # Specificity (True Negative Rate)
        specificity = recall_score(y_true, y_pred, pos_label=0)
        all_specificities.append(specificity)

        # PR-AUC (Average Precision)
        pr_auc = average_precision_score(y_true, y_scores)
        all_pr_aucs.append(pr_auc)

        # Matthews Correlation Coefficient (MCC)
        mcc = matthews_corrcoef(y_true, y_pred)
        all_mccs.append(mcc)

        print(f" Fold {fold + 1} AUROC: {auroc:.4f}")
        print(f" Fold {fold + 1} Balanced Acc: {bal_acc:.4f}")

    mean_auroc = np.mean(all_aurocs)
    std_auroc = np.std(all_aurocs)

    mean_bal_acc = np.mean(all_bal_accs)
    std_bal_acc = np.std(all_bal_accs)

    mean_f1 = np.mean(all_f1s)
    std_f1 = np.std(all_f1s)

    mean_sensitivity = np.mean(all_sensitivities)
    std_sensitivity = np.std(all_sensitivities)

    mean_specificity = np.mean(all_specificities)
    std_specificity = np.std(all_specificities)

    mean_pr_auc = np.mean(all_pr_aucs)
    std_pr_auc = np.std(all_pr_aucs)

    mean_mcc = np.mean(all_mccs)
    std_mcc = np.std(all_mccs)

    print(f"\n Mean AUROC = {mean_auroc:.4f} ± {std_auroc:.4f}")
    print(f" Mean Balanced Accuracy = {mean_bal_acc:.4f} ± {std_bal_acc:.4f}")
    print(f"\n Mean F1 = {mean_f1:.4f} ± {std_f1:.4f}")
    print(f"\n Mean sensitivity = {mean_sensitivity:.4f} ± {std_sensitivity:.4f}")
    print(f"\n Mean specificity = {mean_specificity:.4f} ± {std_specificity:.4f}")
    print(f"\n Mean PR-AUC = {mean_pr_auc:.4f} ± {std_pr_auc:.4f}")
    print(f"\n Mean MCC= {mean_mcc:.4f} ± {std_mcc:.4f}")

    return (
        all_aurocs, mean_auroc, std_auroc,
        all_bal_accs, mean_bal_acc, std_bal_acc
    )

In [16]:
results = cross_validate_stratified( GAT, 
    bbbp_dataset,
    in_channels,
    edge_attr_dim,
    out_channels=1,
    best_lr=config[0],
    best_batch_size=config[1],
    epochs=100,
    k=5
)


 Fold 1
 Fold 1 AUROC: 0.8398
 Fold 1 Balanced Acc: 0.7772

 Fold 2
 Fold 2 AUROC: 0.8553
 Fold 2 Balanced Acc: 0.7812

 Fold 3
 Fold 3 AUROC: 0.8431
 Fold 3 Balanced Acc: 0.7488

 Fold 4
 Fold 4 AUROC: 0.8812
 Fold 4 Balanced Acc: 0.7784

 Fold 5
 Fold 5 AUROC: 0.9106
 Fold 5 Balanced Acc: 0.7533

 Mean AUROC = 0.8660 ± 0.0266
 Mean Balanced Accuracy = 0.7678 ± 0.0138

 Mean F1 = 0.8922 ± 0.0133

 Mean sensitivity = 0.8949 ± 0.0370

 Mean specificity = 0.6407 ± 0.0597

 Mean PR-AUC = 0.9436 ± 0.0178

 Mean MCC= 0.5435 ± 0.0219


# RG

In [17]:

def smiles_to_rgdata(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  
    data = reduce_graph_from_mol_oh(mol)  # pytorch geometric RG graph from mol 
    data.y = torch.tensor([label], dtype=torch.float) #add label 
    return data

def dataframe_to_rg_pyg_dataset(df, smiles_col='nonstereo_aromatic_smiles', label_col='label'):
    data_list = []

    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        label = row[label_col]
        data = smiles_to_rgdata(smiles, label)
        if data is not None:
            #add smiles to dataset 
            data.smiles = smiles            
            data_list.append(data)

    return data_list

In [18]:
bbbp_rg_dataset = dataframe_to_rg_pyg_dataset(dataset, 'smiles', 'p_np')

[10:58:59] Explicit valence for atom # 1 N, 4, is greater than permitted
[10:58:59] WARNING: not removing hydrogen atom without neighbors
[10:58:59] Explicit valence for atom # 6 N, 4, is greater than permitted
[10:59:00] WARNING: not removing hydrogen atom without neighbors
[10:59:00] WARNING: not removing hydrogen atom without neighbors
[10:59:00] WARNING: not removing hydrogen atom without neighbors
[10:59:01] WARNING: not removing hydrogen atom without neighbors
[10:59:01] WARNING: not removing hydrogen atom without neighbors
[10:59:01] WARNING: not removing hydrogen atom without neighbors
[10:59:01] Explicit valence for atom # 6 N, 4, is greater than permitted
[10:59:01] WARNING: not removing hydrogen atom without neighbors
[10:59:01] WARNING: not removing hydrogen atom without neighbors
[10:59:02] WARNING: not removing hydrogen atom without neighbors
[10:59:02] WARNING: not removing hydrogen atom without neighbors
[10:59:02] Explicit valence for atom # 11 N, 4, is greater than pe

In [19]:

model, config, loss = train_eval_model(GAT, bbbp_rg_dataset, bbbp_rg_dataset[0].x.size(-1), bbbp_rg_dataset[0].edge_attr.size(-1), 1, grid)

print(config )


(0.0005, 32)


In [20]:
result = cross_validate_stratified(
    GAT, 
    bbbp_rg_dataset,
    in_channels,
    edge_attr_dim,
    out_channels=1,
    best_lr=config[0],
    best_batch_size=config[1],
    epochs=100,
    k=5
)


 Fold 1
 Fold 1 AUROC: 0.8498
 Fold 1 Balanced Acc: 0.7945

 Fold 2
 Fold 2 AUROC: 0.8153
 Fold 2 Balanced Acc: 0.7444

 Fold 3
 Fold 3 AUROC: 0.8438
 Fold 3 Balanced Acc: 0.7436

 Fold 4
 Fold 4 AUROC: 0.8461
 Fold 4 Balanced Acc: 0.7736

 Fold 5
 Fold 5 AUROC: 0.8769
 Fold 5 Balanced Acc: 0.7963

 Mean AUROC = 0.8464 ± 0.0196
 Mean Balanced Accuracy = 0.7705 ± 0.0231

 Mean F1 = 0.8986 ± 0.0160

 Mean sensitivity = 0.9083 ± 0.0275

 Mean specificity = 0.6326 ± 0.0375

 Mean PR-AUC = 0.9358 ± 0.0074

 Mean MCC= 0.5571 ± 0.0543


# PPGAT

In [21]:

def smiles_to_rgnn_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None  
    G = mol_to_graph(mol)   # mol to networkx 
    data = graph_to_pyg_oh(G)  # networx graph to pytorch geometric graph
    data.y = torch.tensor([label], dtype=torch.float) #add label 


    pharma_index, new_edge_index, new_edge_attr = mol_to_pool_idx(mol)
    data.pharma_index = pharma_index
    data.new_edge_index = new_edge_index
    data.new_edge_attr = new_edge_attr

 

    return data

def dataframe_to_rgnn_pyg_dataset(df, smiles_col='nonstereo_aromatic_smiles', label_col='label'):
    data_list = []

    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        label = row[label_col]
        data = smiles_to_rgnn_data(smiles, label)
        if data is not None:
            data.smiles = smiles
            data_list.append(data)

    return data_list

In [22]:
bbbp_ppgat_dataset = dataframe_to_rgnn_pyg_dataset(dataset, 'smiles', 'p_np')

[11:10:20] Explicit valence for atom # 1 N, 4, is greater than permitted
[11:10:20] WARNING: not removing hydrogen atom without neighbors
[11:10:20] Explicit valence for atom # 6 N, 4, is greater than permitted
[11:10:20] WARNING: not removing hydrogen atom without neighbors
[11:10:20] WARNING: not removing hydrogen atom without neighbors
[11:10:21] WARNING: not removing hydrogen atom without neighbors
[11:10:21] WARNING: not removing hydrogen atom without neighbors
[11:10:21] WARNING: not removing hydrogen atom without neighbors
[11:10:21] WARNING: not removing hydrogen atom without neighbors
[11:10:21] Explicit valence for atom # 6 N, 4, is greater than permitted
[11:10:22] WARNING: not removing hydrogen atom without neighbors
[11:10:22] WARNING: not removing hydrogen atom without neighbors
[11:10:22] WARNING: not removing hydrogen atom without neighbors
[11:10:22] WARNING: not removing hydrogen atom without neighbors
[11:10:22] Explicit valence for atom # 11 N, 4, is greater than pe

In [23]:
model, config, loss = train_eval_model(PPGAT, bbbp_ppgat_dataset, bbbp_ppgat_dataset[0].x.size(-1), bbbp_ppgat_dataset[0].edge_attr.size(-1), 1, grid)
print(config)

(0.0005, 16)


In [24]:
results = cross_validate_stratified(
    PPGAT, 
    bbbp_ppgat_dataset,
    in_channels,
    edge_attr_dim,
    out_channels=1,
    best_lr=config[0],
    best_batch_size=config[1],
    epochs=100,
    k=5
)


 Fold 1
 Fold 1 AUROC: 0.8876
 Fold 1 Balanced Acc: 0.7716

 Fold 2
 Fold 2 AUROC: 0.8510
 Fold 2 Balanced Acc: 0.8021

 Fold 3
 Fold 3 AUROC: 0.8648
 Fold 3 Balanced Acc: 0.8057

 Fold 4
 Fold 4 AUROC: 0.8612
 Fold 4 Balanced Acc: 0.7728

 Fold 5
 Fold 5 AUROC: 0.8787
 Fold 5 Balanced Acc: 0.7975

 Mean AUROC = 0.8687 ± 0.0130
 Mean Balanced Accuracy = 0.7899 ± 0.0147

 Mean F1 = 0.9106 ± 0.0063

 Mean sensitivity = 0.9244 ± 0.0135

 Mean specificity = 0.6555 ± 0.0343

 Mean PR-AUC = 0.9420 ± 0.0096

 Mean MCC= 0.6024 ± 0.0253
